# MolChat P2 — LoRA fine-tune Qwen2.5-0.5B-Instruct (Colab T4)

Fine-tunes a small instruct model on the tool-grounded molecular QA dataset,
evaluates on held-out molecules, and saves a LoRA adapter. Runtime: **T4 GPU**.
See `p2/README.md` for quantization + serving after this.

In [ ]:
!pip -q install "transformers>=4.44" "peft>=0.13" "trl>=0.11" "datasets>=2.20" "accelerate>=0.34" rdkit faiss-cpu

In [ ]:
# Get the repo (private). Set a token, or upload the MolChat/ folder to Colab.
# import os; os.environ['GH_TOKEN']='...'
# !git clone https://$GH_TOKEN@github.com/junghyun-han/MolChat.git
%cd MolChat

In [ ]:
# 2-a: (re)build the tool-grounded dataset
!python scripts/build_dataset.py --out data/qa
!head -n 2 data/qa/train.jsonl

In [ ]:
# 2-b: LoRA fine-tune (a few minutes on a T4)
!python p2/train_lora.py --data data/qa --out p2/out/molchat-qwen-lora --epochs 3

In [ ]:
# Quick inference with the fine-tuned adapter
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = 'Qwen/Qwen2.5-0.5B-Instruct'
tok = AutoTokenizer.from_pretrained('p2/out/molchat-qwen-lora')
model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(base), 'p2/out/molchat-qwen-lora')
msgs = [{'role': 'user', 'content': 'Is diazepam likely to cross the blood-brain barrier?'}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt')
out = model.generate(ids, max_new_tokens=120, do_sample=False)
print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

## Next (on Mac or here): quantize + serve
Follow `p2/README.md` sections 2-c and 2-d: merge the adapter, convert to GGUF,
quantize to Q4_K_M with llama.cpp, serve with Ollama, then point the MolChat
agent at it with `MOLCHAT_LLM=local`.